# 04 · Seleção do vencedor e promoção

**Objetivo.** Decidir, de forma automática e auditável, se o vencedor do ciclo substitui o modelo vigente.

**Entradas.**
- `data/processed/competicao.json`: vencedor e `model_uri` no MLflow.
- `data/processed/drift.json`: status de drift do ciclo.
- `models/modelo_vigente.json`: manifesto do modelo vigente, se existir.

**Saídas.**
- Se houver promoção: `models/versoes/<model_id>/modelo.joblib` e `manifesto.json` (nova versão, nunca sobrescrita), e `models/modelo_vigente.json` atualizado.
- Em todo ciclo: uma linha nova em `models/historico_promocoes.jsonl` e `reports/evidencias/<cycle_id>/04_promocao.json`.
- MLflow: run filho `promocao` e tags de decisão no run pai e no run do vencedor.

**Regra de promoção** (direção `maximize`):

```text
promover  ⇔  F1_desafiante(validação) ≥ F1_vigente(validação) + promotion.minimum_improvement
```

- **Mesma régua.** O modelo vigente é **reavaliado na validação do ciclo atual**. Comparar com o F1 que ele obteve no ciclo dele mediria populações diferentes. Como a partição é estável por ID, o vigente nunca treinou com essas linhas.
- **Primeiro ciclo.** Sem modelo vigente, o vencedor é promovido como modelo inicial.
- **Drift severo.** Com `drift.relax_promotion_on_severe`, a melhoria mínima passa a 0: basta o desafiante não ser pior.
- **Desligada.** Com `promotion.enabled: false`, o vigente é sempre mantido.
- **Teste.** O conjunto de teste só é usado **depois** da decisão, para a avaliação final do modelo que ficou em produção.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

import mlflow
import mlflow.sklearn

from src import drift, modelagem as mdl, preprocessamento as pp, promocao
from src.utils import configure_mlflow, cycle_run, load_config, load_cycle

pd.set_option("display.max_columns", 50)
cfg = load_config(ROOT)
TARGET = cfg["data"]["target"]

In [2]:
ctx = load_cycle(cfg, ROOT)
configure_mlflow(cfg, ROOT)
dataset = pp.load_processed(cfg, ctx, ROOT)
competition = mdl.load_competition(cfg, ctx, ROOT)
drift_report = drift.load_drift(cfg, ctx, ROOT)
metric, direction = cfg["evaluation"]["primary_metric"], cfg["evaluation"]["direction"]

winner = next(m for m in competition["models"] if m["label"] == competition["winner"])
challenger = mlflow.sklearn.load_model(winner["model_uri"])
X_valid, y_valid = pp.split_xy(dataset, "validacao", winner["features"], cfg)
challenger_metric = mdl.evaluate(challenger, X_valid, y_valid, cfg)[metric]

assert abs(challenger_metric - winner["validation"][metric]) < 1e-12, "modelo recarregado do MLflow reproduz a metrica"
print(f"Desafiante: {winner['label']} ({winner['algorithm']}), {metric} = {challenger_metric:.4f}")
print("Drift do ciclo:", drift_report["status"])

Desafiante: modelo_B (random_forest), f1 = 0.5410
Drift do ciclo: estavel


## 1. Modelo vigente reavaliado na validação atual

In [3]:
active = promocao.load_active_model(cfg, ROOT)
if active is None:
    champion_metric = None
    print("Não há modelo vigente.")
else:
    champion_bundle, champion_manifest = active
    X_valid_champion, _ = pp.split_xy(dataset, "validacao", champion_bundle["features"], cfg)
    champion_eval = mdl.evaluate(champion_bundle["pipeline"], X_valid_champion, y_valid, cfg)
    champion_metric = champion_eval[metric]
    print(f"Vigente: {champion_manifest['model_id']}")
    print(f"  {metric} registrado no ciclo dele : {champion_manifest['metric_value']:.4f}")
    print(f"  {metric} na validação deste ciclo : {champion_metric:.4f}")

Vigente: ciclo_20261116T223751Z_a2d3d2d2__random_forest
  f1 registrado no ciclo dele : 0.5439
  f1 na validação deste ciclo : 0.5403


## 2. Decisão

In [4]:
decision = promocao.decide_promotion(challenger_metric, champion_metric, cfg, severe_drift=drift_report["severe"])
display(pd.Series(decision.to_dict(), name="decisão").to_frame())

,decisão
decision,mantido
challenger_metric,0.541009
champion_metric,0.540319
minimum_improvement,0.01
threshold_relaxed_by_drift,False
reason,reprovado: f1_desafiante 0.5410 vs f1_vigente ...
promoted,False


## 3. Persistência e avaliação final no teste

Se houver promoção, o pacote do modelo é gravado como uma versão nova. O pacote contém o pipeline (pré-processador + estimador), a lista de atributos eleitos e os metadados. Em seguida, o modelo que ficou em produção é avaliado **uma única vez** no conjunto de teste, apenas para registro.

In [5]:
if decision.promoted:
    model_id = promocao.model_id_for(ctx, winner["algorithm"])
    bundle = {"pipeline": challenger, "features": winner["features"], "model_id": model_id, "target": TARGET,
              "algorithm": winner["algorithm"], "cycle_id": ctx.cycle_id, "data_version": ctx.data_version}
    X_test, y_test = pp.split_xy(dataset, "teste", bundle["features"], cfg)
    test_metrics = mdl.evaluate(challenger, X_test, y_test, cfg)
    manifest = promocao.persist_promoted_model(bundle, promocao.build_manifest(
        model_id=model_id, run_id=winner["run_id"], model_uri=winner["model_uri"], ctx=ctx,
        algorithm=winner["algorithm"], model_label=winner["label"], cfg=cfg, metric_value=challenger_metric,
        test_metrics=test_metrics, features=winner["features"], params=winner["params"], decision=decision), cfg, ROOT)
    print("Promovido:", manifest["model_id"], "->", manifest["artifact_path"])
else:
    bundle, manifest = champion_bundle, champion_manifest
    X_test, y_test = pp.split_xy(dataset, "teste", bundle["features"], cfg)
    test_metrics = mdl.evaluate(bundle["pipeline"], X_test, y_test, cfg)
    print("Mantido:", manifest["model_id"])

display(pd.Series(test_metrics, name=f"teste ({manifest['model_id']})").to_frame())

Mantido: ciclo_20261116T223751Z_a2d3d2d2__random_forest


,teste (ciclo_20261116T223751Z_a2d3d2d2__random_forest)
f1,0.536348
precision,0.474510
recall,0.616718
accuracy,0.777731
roc_auc,0.784697
tn,3055.000000
fp,670.000000
fn,376.000000
tp,605.000000


## 4. Registro da decisão

In [6]:
entry = {**ctx.tags, "batches": ctx.batches, "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
         "winner": winner["label"], "winner_algorithm": winner["algorithm"], "winner_run_id": winner["run_id"],
         **decision.to_dict(), "primary_metric": metric, "drift_status": drift_report["status"],
         "active_model_id": manifest["model_id"], "active_model_test_metrics": test_metrics}
promocao.record_decision(entry, ctx, cfg, ROOT)

with cycle_run(ctx, cfg, stage="promocao", root=ROOT):
    mlflow.set_tags({"promotion_decision": decision.decision, "winner": winner["label"],
                     "active_model_id": manifest["model_id"], "drift_status": drift_report["status"]})
    mlflow.log_params({"minimum_improvement_config": cfg["promotion"]["minimum_improvement"],
                       "minimum_improvement_applied": decision.minimum_improvement,
                       "threshold_relaxed_by_drift": decision.threshold_relaxed_by_drift})
    mlflow.log_metric(f"challenger_val_{metric}", challenger_metric)
    if champion_metric is not None:
        mlflow.log_metric(f"champion_val_{metric}", champion_metric)
    mlflow.log_metrics({f"active_test_{k}": v for k, v in test_metrics.items()})
    mlflow.log_dict(entry, "decisao_promocao.json")

client = mlflow.MlflowClient()
client.set_tag(winner["run_id"], "promotion_decision", decision.decision)
client.set_tag(ctx.parent_run_id, "promotion_decision", decision.decision)
client.set_tag(ctx.parent_run_id, "active_model_id", manifest["model_id"])
print(decision.decision, "|", decision.reason)

mantido | reprovado: f1_desafiante 0.5410 vs f1_vigente 0.5403 + 0.0100


## Conclusões

- A decisão é automática. Ela segue a regra configurada e mede desafiante e vigente na mesma validação.
- Toda decisão, promovida ou não, fica no histórico, nas evidências e no MLflow.
- Versões promovidas nunca são sobrescritas: cada uma tem diretório próprio com SHA-256 registrado no manifesto. A versão anterior continua recuperável em `models/versoes/` e no MLflow.
- O teste foi usado só depois da decisão, para a avaliação final do modelo em produção.

**Próximo passo:** `05_previsoes.ipynb` gera previsões com o modelo vigente.